In [ ]:
import anthropic
import json

from statistics import mean

from dotenv import load_dotenv
load_dotenv()

def add_user_messages(messages, message):
    messages.append({"role": "user", "content": message})

def add_assistant_messages(messages, message):
    messages.append({"role": "assistant", "content": message})
    
def chat(messages, system=None, temperature=1.0, stop_sequences=None):
    parameters = {
        "model": "claude-sonnet-4-6",
        "max_tokens": 1024,
        "messages": messages,
        "temperature": temperature
    }
    
    if system:
        parameters["system"] = system
    
    if stop_sequences:
        parameters["stop_sequences"] = stop_sequences

    client = anthropic.Anthropic()
    response = client.messages.create(**parameters)
    return response.content[0].text

def dataset(messages):
    client = anthropic.Anthropic()
    with client.messages.stream(
        model="claude-haiku-4-5",
        max_tokens=1024,
        messages=messages,
        system="""
            Generate JSON array of tasks to measure prompt efficieny. 
            Tasks have to cover next areas: Python, Regex, AWS configuration.
            Output should be without any markdown or code block.

            Example output:
            ```json
            [
                {
                    "task": "Description of task",
                }
            ]
            ```

            Please generate 3 objects. 
            """,
    ) as stream:
        for response in stream.text_stream:
            pass
    return stream.get_final_message().content[0].text

def run_prompt(test_case):
    messages = []
    prompt = f"""
        Please solve the following task: {test_case}
    """
    add_user_messages(messages, prompt)
    output = chat(messages)
    return output

def run_test_case(test_case):
    output = run_prompt(test_case)
    
    result = {
        "task": test_case,
        "output": output,
        "grade": grade_by_model(test_case, output)
    }

    print(result)
    return result

def run_eval(dataset):
    results = []
    for item in dataset:
        test_case = item["task"]
        print(f"Running test case: {test_case}")
        result = run_test_case(test_case)
        print(f"Result: {result}")
        results.append(result)
    
    average_score = mean([result["grade"]["grade"] for result in results])
    print(f"Average score: {average_score}")

    return results

def grade_by_model(test_case, output):
    messages = []
    prompt = f"""
        You are an expert code reviewer. Evaluate this AI-generated solution.
        Please grade the following output for the task: 
        Task: {test_case}
        Output: {output}

        Provide your evaluation as JSON object:
        - grade: integer from 0 to 10, where 10 is the best grade.
        - comment: short explanation of the grade.

        Output should be without any markdown or code block, only JSON object. 
        Don't wrap output in any markdown. Don't use any backticks.
    """
    add_user_messages(messages, prompt)
    grade = chat(messages, system=""" 
                 You are an expert code reviewer. 
                 Output should be only JSON object with grade and comment. 
                 No markdown or code blocks. Don't wrap output in any markdown.
                 """)
    
    print(f"Grade response: {grade}")
    return json.loads(grade)

In [23]:
with open("dataset.json", "r") as f:
    dataset = json.loads(json.load(f))
results = run_eval(dataset)

print(json.dumps(results, indent=4))

Running test case: Write a Python function that takes a list of integers and returns a new list containing only the even numbers, sorted in descending order. The function should handle empty lists and use list comprehension.
Grade response: {"grade": 10, "comment": "The solution is excellent. It correctly implements the required function using list comprehension, handles empty lists, sorts in descending order, and includes proper type hints and docstring. The test suite is comprehensive, covering edge cases like negative numbers, zero, single elements, and all-even/no-even lists. All tests pass. The code is clean, readable, and well-documented."}
{'task': 'Write a Python function that takes a list of integers and returns a new list containing only the even numbers, sorted in descending order. The function should handle empty lists and use list comprehension.', 'output': '# Even Numbers Filter and Sort Function\n\n## Solution\n\n```python\ndef filter_even_descending(numbers: list[int]) 